**1\. Get the top 3 most expensive products per category.**  

> Using `Production.Product` and `Production.ProductSubcategory`

In [ ]:
SELECT 
  p.Name AS ProductName,
  ps.Name AS Subcategory,
  p.ListPrice,
  RANK() OVER(PARTITION BY ps.Name ORDER BY p.ListPrice DESC) AS rnk
FROM Production.Product p
JOIN Production.ProductSubcategory ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
WHERE p.ListPrice > 0


**2\. Show each employee's salary history with difference from previous record.**  

> Using `HumanResources.EmployeePayHistory`

In [ ]:
SELECT 
  BusinessEntityID,
  RateChangeDate,
  Rate,
  LAG(Rate) OVER(PARTITION BY BusinessEntityID ORDER BY RateChangeDate) AS PrevRate,
  Rate - LAG(Rate) OVER(PARTITION BY BusinessEntityID ORDER BY RateChangeDate) AS RateDiff
FROM HumanResources.EmployeePayHistory


3:

### **3\. Find the first and last product price for each product.**

> Using `Production.ProductListPriceHistory`

In [ ]:
SELECT 
  ProductID,
  ListPrice,
  StartDate,
  FIRST_VALUE(ListPrice) OVER(PARTITION BY ProductID ORDER BY StartDate) AS FirstPrice,
  LAST_VALUE(ListPrice) OVER(PARTITION BY ProductID ORDER BY StartDate ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING) AS LastPrice
FROM Production.ProductListPriceHistory


4-

### **4\. Compare number of orders per year by territory using PIVOT.**

> Using `Sales.SalesOrderHeader`

In [ ]:
SELECT *
FROM (
  SELECT 
    TerritoryID,
    YEAR(OrderDate) AS OrderYear
  FROM Sales.SalesOrderHeader
) AS src
PIVOT (
  COUNT(OrderYear)
  FOR OrderYear IN ([2011], [2012], [2013], [2014])
) AS p


5-

### **5\. DENSE\_RANK() — No gaps in ranking**

> Rank employees by hire date  
> **(Window Function: DENSE\_RANK)**

In [ ]:

SELECT 
  BusinessEntityID,
  HireDate,
  DENSE_RANK() OVER(ORDER BY HireDate) AS HireOrder
FROM HumanResources.Employee;



6-

### **6\. Show running total of product inventory by product.**

> Using `Production.ProductInventory`

In [ ]:
SELECT 
  ProductID,
  LocationID,
  Shelf,
  Quantity,
  SUM(Quantity) OVER(PARTITION BY ProductID ORDER BY LocationID) AS RunningTotal
FROM Production.ProductInventory


7-

### **7\. Find customers with the highest total sales.**

> Using `Sales.SalesOrderHeader`

In [ ]:
SELECT 
  CustomerID,
  SUM(TotalDue) AS TotalSales,
  RANK() OVER(ORDER BY SUM(TotalDue) DESC) AS SalesRank
FROM Sales.SalesOrderHeader
GROUP BY CustomerID


8-

### **8\. Use GROUPING SETS to compare sales by salesperson and customer.**

> Using `Sales.SalesOrderHeader`

In [ ]:
SELECT 
  GROUPING_ID(SalesPersonID, CustomerID) AS grouping_id,
  SalesPersonID,
  CustomerID,
  SUM(TotalDue) AS TotalSales
FROM Sales.SalesOrderHeader
GROUP BY GROUPING SETS (
  (SalesPersonID, CustomerID),
  (SalesPersonID),
  (CustomerID),
  ()
)


  

### **9\. Get product reviews and rank them per product.**

> Using `Production.ProductReview`

In [ ]:
SELECT 
  ProductID,
  ReviewerName,
  Rating,
  RANK() OVER(PARTITION BY ProductID ORDER BY Rating DESC) AS RankPerProduct
FROM Production.ProductReview


  

### **10\. Show employee count per department and shift using CUBE.**

> Using `HumanResources.EmployeeDepartmentHistory`

In [ ]:
SELECT 
  DepartmentID,
  ShiftID,
  COUNT(BusinessEntityID) AS EmployeeCount,
  GROUPING_ID(DepartmentID, ShiftID) AS grouping_id
FROM HumanResources.EmployeeDepartmentHistory
GROUP BY CUBE(DepartmentID, ShiftID)



### 🔍 Discussion: Why These Queries Were Considered Special

**1. RANK() – Top Products by Subcategory**  
This query ranks products within each subcategory by price. It’s special because it helps identify top-performing or highest-priced items using a single SQL statement with partitioning.

**2. LAG() – Rate Change Over Time**  
This query compares the current rate to the previous rate for each employee. It’s special because it uses window functions to track changes over time without a self-join.

**3. FIRST_VALUE() & LAST_VALUE() – Product Price Evolution**  
By pulling the first and last prices from the history, this query gives insights into how pricing has changed over time. It’s special because it allows easy comparison across timelines using built-in window functions.

**4. PIVOT – Orders by Territory and Year**  
This query uses the PIVOT operator to reshape order data across years and territories. It’s special because it transforms long data into a readable wide format, ideal for reports.

**5. DENSE_RANK() – Employee Hiring Order**  
This query ranks employees by hire date. DENSE_RANK ensures there are no gaps in ranking if employees were hired on the same day. It’s special for fairness in HR reporting.

**6. SUM() OVER() – Running Inventory Totals**  
This query calculates running totals of product quantities across locations. It’s special for inventory management and tracking accumulation trends over dimensions.

**7. RANK() – Customer Sales Ranking**  
This query ranks customers based on their total purchases. It’s special because it quickly highlights high-value customers using window-based aggregate logic.

**8. GROUPING SETS + GROUPING_ID() – Flexible Sales Summarization**  
This query generates sales totals by different combinations (salesperson, customer, etc.). It’s special because it replaces multiple GROUP BYs with one smart solution, and GROUPING_ID helps identify the summary level.

**9. RANK() – Product Review Ranking**  
This query ranks reviews per product based on rating. It’s special for understanding which reviews were most favorable, useful in feedback analysis or e-commerce sorting.

**10. GROUPING_ID() + CUBE() – Departmental Employee Counts**  
This query groups employee counts by department and shift using CUBE, and tags each level using GROUPING_ID. It’s special for multidimensional reporting in HR or operations dashboards.
